In [13]:
#Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [15]:
# import data
data = pd.read_csv('https://raw.githubusercontent.com/dvasiliu/AML/main/Data%20Sets/housing.csv?raw=true')

In [11]:
data

,town,tract,longitude,latitude,crime,residential,industrial,river,nox,rooms,older,distance,highway,tax,ptratio,lstat,cmedv
0,Nahant,2011,-70.955002,42.255001,0.00632,18.0,2.31,no,0.538,6.575,65.199997,4.0900,1,296,15.300000,4.98,24.000000
1,Swampscott,2021,-70.949997,42.287498,0.02731,0.0,7.07,no,0.469,6.421,78.900002,4.9671,2,242,17.799999,9.14,21.600000
2,Swampscott,2022,-70.935997,42.283001,0.02729,0.0,7.07,no,0.469,7.185,61.099998,4.9671,2,242,17.799999,4.03,34.700001
3,Marblehead,2031,-70.928001,42.292999,0.03237,0.0,2.18,no,0.458,6.998,45.799999,6.0622,3,222,18.700001,2.94,33.400002
4,Marblehead,2032,-70.921997,42.298000,0.06905,0.0,2.18,no,0.458,7.147,54.200001,6.0622,3,222,18.700001,5.33,36.200001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,Winthrop,1801,-70.986000,42.231201,0.06263,0.0,11.93,no,0.573,6.593,69.099998,2.4786,1,273,21.000000,9.67,22.400000
502,Winthrop,1802,-70.990997,42.227501,0.04527,0.0,11.93,no,0.573,6.120,76.699997,2.2875,1,273,21.000000,9.08,20.600000
503,Winthrop,1803,-70.994797,42.226002,0.06076,0.0,11.93,no,0.573,6.976,91.000000,2.1675,1,273,21.000000,5.64,23.900000
504,Winthrop,1804,-70.987503,42.223999,0.10959,0.0,11.93,no,0.573,6.794,89.300003,2.3889,1,273,21.000000,6.48,22.000000


In [16]:
x = data.drop('river',axis=1).loc[:,'crime':'lstat'].values
y = data['cmedv'].values

- We want to apply ENET to predict y
- We want to implement more robust gradient descent flovors, such ADAM
- We also want to experiment with feature engineering, such as polynomial features

In [17]:
def MSE(w,b,x,y):
    predictions = x@w + b
    errors = y-predictions
    return np.mean(errors**2)

In [18]:
def gradient_enet(w,b,x,y,alpha,lam):
    predictions = x@w + b
    errors = y - predictions
    return -2/len(y)*errors@x + alpha*(lam*np.sign(w) + (1-lam)*np.array(w)), -2/len(y)*sum(errors) # here we did not want to regularize the intercept term

In [19]:
def zscore(x):
    return (x- np.mean(x,axis=0))/np.std(x,axis=0)

In [35]:
xscaled = zscore(x)

In [36]:
# we want the "optimal" value for the weights
maxiter = 1000
learning_rate = 0.01
# we want some stopping criteria, for example if the weights are no longer significantly updated
# we measure the difference between w_new and w_old vs epsilon -> tolerance for deciding convergence.
tol = 1e-8
w_old = np.zeros(x.shape[1])
b_old = 0
alpha = 0.01
lam = 0.5

In [37]:
# how to implement the gradient descent
for it in range(maxiter):
    gw, gb =  gradient_enet(w_old,b_old,xscaled,y,alpha,lam)
    # here we update the weights by using the plain "vanilla" gradient descent
    w_new = w_old - learning_rate*gw
    b_new = b_old - learning_rate*gb

    if max(abs(w_new-w_old))<tol:
        print("The algorithm has converged!")
        break

    w_old = w_new
    b_old = b_new
    if (it+1)%100==0:
        print('The MSE is : '+str(MSE(w_old,b_old,xscaled,y)))
w_trained = w_old
b_trained = b_old

The MSE is : 33.79592688400354
The MSE is : 23.851973440910808
The MSE is : 23.24858063927019
The MSE is : 22.99984139944088
The MSE is : 22.85350113556467
The MSE is : 22.76072692830853
The MSE is : 22.698637247133153
The MSE is : 22.65525213903548
The MSE is : 22.623907845392818
The MSE is : 22.600690113675277


In [25]:
# critical thinking question: are the MSE and the weights optimal?
# we can do a quick comparison with SKlearn
from sklearn.linear_model import ElasticNet
model = ElasticNet(alpha=0.01, fit_intercept=True)
model.fit(xscaled,y)
val = np.mean((y-model.predict(xscaled))**2)
print(f'The resulting MSE is : {val}')

The resulting MSE is : 22.53033751854787


In [44]:
# we infer that it may be a good idea to implement stochastic mini-batches and a max 1000 epochs by using an adaptive method such ADAM
# we want the "optimal" value for the weights
# we want the "optimal" value for the weights
epochs = 1000
batch_size = 20
# we want some stopping criteria, for example if the weights are no longer significantly updated
# we measure the difference between w_new and w_old vs epsilon -> tolerance for deciding convergence.
tol = 1e-8
done = False
eta = 0.01
epsilon = 1e-8
sw = 0
vw = 0
sb = 0
vb = 0
beta1 = 0.9
beta2 = 0.9
t = 0

In [45]:
# how to implement the mini-batch adaptive gradient descent
for epoch in range(epochs):
    ind = np.random.permutation(range(len(x)))
    batches = np.array_split(ind,len(x)//batch_size)
    for ind in batches:
        gw, gb =  gradient_enet(w_old,b_old,xscaled[ind],y[ind],alpha,lam)

        # here below we have the updates for the weights
        vw = beta1*vw + (1-beta1)*gw
        sw = beta2*sw +(1-beta2)*sum(gw**2)
        vw = v/(1-beta1**(t+1))
        sw = s/(1-beta2**(t+1))
        learning_rate_w = eta/(np.sqrt(sw)+epsilon)
        w_new = w_old - learning_rate_w*vw

        # we also need the updates for the bias term
        vb = beta1*vb + (1-beta1)*gb
        sb = beta2*sb +(1-beta2)*(gb**2)
        vb = vb/(1-beta1**(t+1))
        sb = sb/(1-beta2**(t+1))
        learning_rate_b = eta/(np.sqrt(sb)+epsilon)
        b_new = b_old - learning_rate_b*vb
        
        
        if max(abs(w_new-w_old))<tol:
            done = True
            break
        t += 1
        w_old = w_new
        b_old = b_new
    if (epoch+1)%100==0:
        print('The MSE is : '+str(MSE(w_old,b_old,xscaled,y)))
    if done:
        print("The algorithm has converged!")
        break
w_trained = w_new
b_trained = b_new

The algorithm has converged!


In [46]:
MSE(w_old,b_old,xscaled,y)

np.float64(22.600690113675277)

In [47]:
w_trained

array([-1.00867402,  1.01513774, -0.00919068, -1.95693981,  2.6609832 ,
        0.09940315, -3.12174248,  1.96743797, -1.68482404, -2.00718428,
       -4.01220528])

In [48]:
model.coef_

array([-1.04414275,  1.07686118,  0.14654714, -2.01149266,  2.61287422,
        0.12377427, -3.12764859,  2.42901115, -2.19644631, -2.02169818,
       -4.01129134])

In [86]:
def enet_fit(xt, yt, alpha = 0.01, lam = 0.5, maxiter=1000, learning_rate = 0.01, eta =0.01, epochs =1000, beta1 =0.9, beta2=0.9, tol = 1e-8):
    w_old = np.zeros(xt.shape[1])
    b_old = 0
    done = False
    # how to implement the gradient descent
    for it in range(maxiter):
        gw, gb =  gradient_enet(w_old,b_old,xt,yt,alpha,lam)
        # here we update the weights by using the plain "vanilla" gradient descent
        w_new = w_old - learning_rate*gw
        b_new = b_old - learning_rate*gb
    
        if max(abs(w_new-w_old))<tol:
            print("The algorithm has converged!")
            done = True
            break
    
        w_old = w_new
        b_old = b_new
        if (it+1)%100==0:
            print('The MSE is : '+str(MSE(w_old,b_old,xt,yt)))

    if done:
        return w_new, b_new
    else:
        epsilon = 1e-8
        sw = 0
        vw = 0
        sb = 0
        vb = 0
        t = 0
        # how to implement the mini-batch adaptive gradient descent
        for epoch in range(epochs):
            ind = np.random.permutation(range(len(xt)))
            batches = np.array_split(ind,len(xt)//batch_size)
            for ind in batches:
                gw, gb =  gradient_enet(w_old,b_old,xt[ind],yt[ind],alpha,lam)
        
                # here below we have the updates for the weights
                vw = beta1*vw + (1-beta1)*gw
                sw = beta2*sw +(1-beta2)*sum(gw**2)
                vw = v/(1-beta1**(t+1))
                sw = s/(1-beta2**(t+1))
                learning_rate_w = eta/(np.sqrt(sw)+epsilon)
                w_new = w_old - learning_rate_w*vw
        
                # we also need the updates for the bias term
                vb = beta1*vb + (1-beta1)*gb
                sb = beta2*sb +(1-beta2)*(gb**2)
                vb = vb/(1-beta1**(t+1))
                sb = sb/(1-beta2**(t+1))
                learning_rate_b = eta/(np.sqrt(sb)+epsilon)
                b_new = b_old - learning_rate_b*vb
                
                
                if max(abs(w_new-w_old))<tol:
                    done = True
                    break
                t += 1
                w_old = w_new
                b_old = b_new
            if (epoch+1)%100==0:
                print('The MSE is : '+str(MSE(w_old,b_old,xt,yt)))
            if done:
                print("The algorithm has converged!")
                break
        return w_new, b_new
        
    

In [87]:
w, b = enet_fit(xscaled,y)

The MSE is : 33.79592688400354
The MSE is : 23.851973440910808
The MSE is : 23.24858063927019
The MSE is : 22.99984139944088
The MSE is : 22.85350113556467
The MSE is : 22.76072692830853
The MSE is : 22.698637247133153
The MSE is : 22.65525213903548
The MSE is : 22.623907845392818
The MSE is : 22.600690113675277
The algorithm has converged!


In [51]:
MSE(w,b,xscaled,y)

np.float64(22.600790114416323)

## Polynomial Features

In [55]:
# we will show how to make quadratic features from scratch
# for example if you had two features a and b, you need to add all quadratic interactions a^2, ab and b^2
# we can think of augmenting xscaled with polynomial features
x_poly = xscaled
for i in range(xscaled.shape[1]):
    for j in range(i,xscaled.shape[1]):
        x_poly = np.column_stack([x_poly,xscaled[:,i]*xscaled[:,j]])

In [58]:
xscaled.shape

(506, 11)

In [88]:
w, b = enet_fit(x_poly,y,tol=1e-10)

The MSE is : 26.234268708100736
The MSE is : 19.643675392373012
The MSE is : 16.462656296435885
The MSE is : 14.458991776507858
The MSE is : 13.085061096286973
The MSE is : 12.095260490447282
The MSE is : 11.357251208140102
The MSE is : 10.792914274020646
The MSE is : 10.351155778879711
The MSE is : 9.999196784217526
The algorithm has converged!


In [63]:
# for polynomial features of higher degree we may use sklearn
from sklearn.preprocessing import PolynomialFeatures

In [64]:
# suppose we want cubic featuers
poly = PolynomialFeatures(degree=3)
x_poly_deg3 = poly.fit_transform(xscaled)

In [65]:
x_poly_deg3.shape

(506, 364)

In [75]:
w, b = enet_fit(x_poly_deg3,y,maxiter=5000,learning_rate = 0.0002, eta = 0.001,tol=1e-10)

The MSE is : 179.6472287440831
The MSE is : 113.50618400185901
The MSE is : 84.30008428984155
The MSE is : 67.97214976188697
The MSE is : 57.58069284189917
The MSE is : 50.44375782015734
The MSE is : 45.280193363120226
The MSE is : 41.38800023558775
The MSE is : 38.35141857579802
The MSE is : 35.91134125770492
The MSE is : 33.90078131061462
The MSE is : 32.20882088479723
The MSE is : 30.759370635387725
The MSE is : 29.49908960557444
The MSE is : 28.38931287077121
The MSE is : 27.401526136439227
The MSE is : 26.514306296091718
The MSE is : 25.711153456874758
The MSE is : 24.978973061603448
The MSE is : 24.307310384529
The MSE is : 23.687783069928543
The MSE is : 23.1134966128737
The MSE is : 22.578838157923762
The MSE is : 22.07898865939664
The MSE is : 21.609989008601033
The MSE is : 21.16849599245062
The MSE is : 20.751614987689415
The MSE is : 20.356966426645712
The MSE is : 19.98236506987968
The MSE is : 19.625951501054328
The MSE is : 19.286115146973668
The MSE is : 18.961452458847

In [90]:
# only for a simple example I will use sklearn for scaling data correctly
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
# to do k-fold CV from scratch
# shuffle the data
ind = np.random.permutation(len(x))
# subdivide into k folds
k = 5
folds = np.array_split(ind,k)

# the actual CV loop
mse = []
for fold in folds:
    indtest = ind[fold]
    indtrain = np.delete(ind,testind)
    xtrain = scale.fit_transform(x[indtrain])
    xtest = scale.transform(x[indtest])
    ytrain = y[indtrain]
    ytest = y[indtest]
    w_train, b_train = enet_fit(xtrain,ytrain)
    mse_test = np.mean((ytest-xtest@w_train-b_train)**2)
    print(f'The MSE is : {mse_test}')
    mse.append(mse_test)
    ####

The MSE is : 33.088407300081016
The MSE is : 23.102655603066136
The MSE is : 22.506952036494837
The MSE is : 22.259615078060797
The MSE is : 22.113105178197124
The MSE is : 22.019510818990568
The MSE is : 21.956409513066976
The MSE is : 21.912042106187073
The MSE is : 21.879819587346383
The MSE is : 21.85584451152121
The algorithm has converged!
The MSE is : 19.184588121608627
The MSE is : 33.088407300081016
The MSE is : 23.102655603066136
The MSE is : 22.506952036494837
The MSE is : 22.259615078060797
The MSE is : 22.113105178197124
The MSE is : 22.019510818990568
The MSE is : 21.956409513066976
The MSE is : 21.912042106187073
The MSE is : 21.879819587346383
The MSE is : 21.85584451152121
The algorithm has converged!
The MSE is : 28.739754757705406
The MSE is : 33.088407300081016
The MSE is : 23.102655603066136
The MSE is : 22.506952036494837
The MSE is : 22.259615078060797
The MSE is : 22.113105178197124
The MSE is : 22.019510818990568
The MSE is : 21.956409513066976
The MSE is : 21.

In [92]:
mse

[np.float64(19.184588121608627),
 np.float64(28.739754757705406),
 np.float64(24.257698863481995),
 np.float64(15.075856705641417),
 np.float64(26.230857417405243)]